In [0]:
%sql
CREATE OR REPLACE TABLE electricity.gold.hourly_grid AS
SELECT
  date_trunc('hour', start_time_utc)                        AS hour_utc,
  from_utc_timestamp(date_trunc('hour', start_time_utc), 'Europe/Helsinki') AS hour_fi,
  ROUND(SUM(CASE WHEN measure = 'consumption' THEN value_mw * resolution_min END)
    / SUM(CASE WHEN measure = 'consumption' THEN resolution_min END),1)  AS consumption_mw,
  ROUND(SUM(CASE WHEN measure = 'production' THEN value_mw * resolution_min END)
    / SUM(CASE WHEN measure = 'production' THEN resolution_min END),1)  AS production_mw,
  ROUND(SUM(CASE WHEN measure = 'wind' THEN value_mw * resolution_min END)
    / SUM(CASE WHEN measure = 'wind' THEN resolution_min END),1)             AS wind_mw,
  SUM(CASE WHEN measure = 'consumption' THEN resolution_min END)      AS consumption_minutes,
  ROUND(consumption_mw - production_mw, 1) AS net_import_mw
FROM electricity.silver.grid_observations
GROUP BY date_trunc('hour', start_time_utc);

In [0]:
%sql
SELECT * FROM electricity.gold.hourly_grid ORDER BY hour_utc DESC LIMIT 10

In [0]:
%sql
CREATE OR REPLACE VIEW electricity.gold.daily_grid AS
SELECT
  CAST(hour_fi AS DATE)                              AS day_fi,
  ROUND(AVG(consumption_mw), 1)                      AS avg_consumption_mw,
  ROUND(AVG(production_mw), 1)                       AS avg_production_mw,
  ROUND(AVG(wind_mw), 1)                             AS avg_wind_mw,
  ROUND(AVG(net_import_mw), 1)                       AS avg_net_import_mw,
  ROUND(MAX(consumption_mw), 1)                      AS peak_consumption_mw,
  ROUND(AVG(wind_mw) / AVG(production_mw) * 100, 1)  AS wind_share_pct,
  COUNT(*)                                           AS hours_in_day
FROM electricity.gold.hourly_grid
GROUP BY CAST(hour_fi AS DATE);

In [0]:
%sql
SELECT * FROM electricity.gold.daily_grid ORDER BY day_fi DESC LIMIT 10;

In [0]:
%sql
SELECT day_fi, hours_in_day
FROM electricity.gold.daily_grid
WHERE hours_in_day <> 24
ORDER BY day_fi;

In [0]:
%sql
SELECT hour_fi FROM electricity.gold.hourly_grid
WHERE CAST(hour_fi AS DATE) = '2024-06-03' ORDER BY hour_fi;